# WO-10 — sample profile page: the stored profile records
Cell 2 checks that the markdown is unchanged by the generator change (template lines recorded per unit), checks that each recorded line sits inside its unit's text, and writes one record per place to `app/static/profiles/`. Cell 3 prints the template view of one place. Spec: `docs/design/dashboard/wo10_sample-profile-page.md`.

In [4]:
%matplotlib inline
# Cell 1 — Setup: wording file, server check, saved ring, flow and river tables
import sys, time, warnings, json
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W
from scripts.edop.whg import wo06_rings as R
from scripts.edop.whg import wo07_rivers as RV
from scripts.edop.whg import wo10_profiles as P

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
RING = R.load_table()
RIV = RV.load_table() if RV.RIVERS_CSV.exists() else None
print('commit', C.commit(), '| server:', W.get_json('/api/health'), '| wording:', W.templates_stamp(), '| ring table:', len(RING), 'basins | river table:', len(RIV) if RIV else 'not built yet', '| places:', len(C.PLACES))


commit 92ae0c0 | server: {'status': 'ok'} | wording: profile_templates.md v22 (last commit fe5b153; edited since: yes); script wo04_fill.py (last commit c1095c9; edited since: no); script wo06_rings.py (last commit 3bfacd6; edited since: yes); script wo07_rivers.py (last commit f153841; edited since: no); script wo07_classes.py (last commit 7090f1c; edited since: no); script wo08_units.py (last commit fe5b153; edited since: no) | ring table: 315 basins | river table: 574 | places: 10


In [5]:
# Cell 2 — The ten places: (a) the markdown must be what wo09_filled_profiles.md holds, except where wording changed since (v21: S5, S6.2, S8.2) and where unbuilt units are hidden, (b) every template line recorded for a unit must sit inside the unit's text (case aside), (c) the records and the response files are written to app/static/profiles/
OLD = (ROOT / 'docs' / 'design' / 'dashboard' / 'wo09_filled_profiles.md').read_text()
t0 = time.perf_counter()
RECS, FACTS, UNITS, L = [], {}, {}, []
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    Un = W.fill_units(F)
    FACTS[name], UNITS[name] = F, Un
    md = W.render_filled_profile(name, F, Un)
    same = md in OLD
    odd = [f'{uid} [{ln["key"]}]' for uid, u in Un.items() if u['ok'] for ln in u['lines'] if ln['out'].lower() not in (u['text'] or '').lower()]
    rec = P.build_record(name, F, Un, time.strftime('%Y-%m-%d'))
    RECS.append(rec)
    n_lines = sum(len(u['lines']) for u in Un.values())
    L.append(f'{name}: markdown identical to the committed file: {same} | units with lines: {sum(1 for u in Un.values() if u["lines"])} ({n_lines} lines) | line not inside its unit text: {odd or "none"} | sections {len(rec["sections"])} | buttons {[b["id"] for b in rec["buttons"]]}')
paths = P.write_records(RECS)
L.append('')
L += [f'{p.relative_to(ROOT)}  {p.stat().st_size:,} bytes' for p in paths]
print('\n'.join(L) + f'\n\n{time.perf_counter() - t0:.0f}s')


Timbuktu: markdown identical to the committed file: False | units with lines: 21 (22 lines) | line not inside its unit text: none | sections 7 | buttons ['S9.3', 'signature']
Tbilisi: markdown identical to the committed file: False | units with lines: 21 (22 lines) | line not inside its unit text: none | sections 7 | buttons ['S9.3', 'signature', 'S9.2']
Santa Fe: markdown identical to the committed file: False | units with lines: 19 (20 lines) | line not inside its unit text: none | sections 7 | buttons ['S9.3', 'signature', 'S9.2']
Kaifeng: markdown identical to the committed file: False | units with lines: 21 (22 lines) | line not inside its unit text: none | sections 7 | buttons ['S9.3', 'signature', 'S9.2']
Lviv: markdown identical to the committed file: False | units with lines: 16 (17 lines) | line not inside its unit text: none | sections 7 | buttons ['S9.3', 'signature', 'S9.2']
Poltava: markdown identical to the committed file: False | units with lines: 17 (18 lines) | line n

In [ ]:
# Cell 3 — The template view of one place, as the stored record has it (change PLACE to look at another)
PLACE = 'Timbuktu'
rec = next(r for r in RECS if r['place'] == PLACE)
out = [f'{rec["place"]} | template v{rec["template_version"]} | basin {rec["hybas_id"]}', '']
for sec in rec['sections']:
    out.append(f'## {sec["title"]}')
    for s in sec['template']:
        if s['type'] == 'text':
            if s['text'].strip():
                out.append('   ' + s['text'].strip())
        elif s['type'] == 'slot':
            out.append('   {' + s['slot'] + '}')
        else:
            out.append(f'   [{s["key"]}] {s["raw"]}')
            for p_ in s['parts']:
                out.append(f'        part [{p_["key"]}] {p_["raw"]}')
    out.append('')
print('\n'.join(out))


In [3]:
# Cell 4 — WO-10a edge cases first: the response of Tristan da Cunha, Alexandria, Utqiagvik and Kaifeng (size, links, and the sentences v21 changed)
import re
L = []
for name in ['Tristan da Cunha', 'Alexandria', 'Utqiagvik', 'Kaifeng']:
    rec = next(r for r in RECS if r['place'] == name)
    resp = P.to_response(rec)
    L.append(f'## {name} | response {len(json.dumps(resp, ensure_ascii=False, indent=1)):,} bytes | keys {list(resp)}')
    if resp['no_basin']:
        L += [f'   {resp["message"]}', '']
        continue
    L.append(f'   links: {len(resp["links"]["ecoregions"])} ecoregion(s), wikipedia {"yes" if resp["links"]["wikipedia"] else "no"}')
    for s in resp['sections']:
        if s['id'] in ('S5', 'S6', 'S8'):
            L.append(f'   [{s["id"]}] {s["text"]}')
    L.append('')
print('\n'.join(L))


## Tristan da Cunha | response 297 bytes | keys ['place', 'lat', 'lon', 'hybas_id', 'generated', 'template_version', 'no_basin', 'message']
   No basin contains this point (structured response `no_basin`; nearest basin: see WO-03 Q6). Every basin-keyed unit is empty.

## Alexandria | response 6,736 bytes | keys ['place', 'lat', 'lon', 'hybas_id', 'generated', 'template_version', 'no_basin', 'sections', 'monthly', 'links']
   links: 3 ecoregion(s), wikipedia yes
   [S5] Around 2000, about 24% of the basin is cropland and 3% is pasture; in the GLC2000 land cover map, the commonest class other than cultivated land is bare areas, at about 54%. Compared with the five basins around it, its basin has more cropland than all but one and more pasture than three.
   [S6] Alexandria's basin is one of six that meet here. None of them drains into it. It drains into none of them. All but one of the basins across its divides are higher. Of the six basins that meet here, it has the most runoff.
   [S8]